# Airbnb Paris – FastText (PCA 30 & 100)
- Freitexte per unsupervised fastText (skipgram, 100d), Spaltennamen-Embedding addiert, LayerNorm je Vektor
- fastText **und** PCA werden nur auf Trainingszeilen gefittet
- `thread=1`, weil fastText sonst Hogwild-parallel und damit nichtdeterministisch ist; der interne RNG ist im Python-Binding auf 0 fixiert und nicht setzbar, die Variation über die Seeds entsteht also über den unterschiedlichen Trainingskorpus

In [ ]:
import os
import numpy as np
import pandas as pd
import fasttext
from sklearn.decomposition import PCA

SEED = int(os.environ.get("SEED", 1))
TEXT_COLS = ["name", "description", "neighborhood_overview", "host_about"]
print("SEED", SEED)

## Basis, Texte & Split laden

In [ ]:
base = pd.read_csv(f"../../data/preprocessed/cleaned_airbnb_paris_seed{SEED}.csv")
texts = pd.read_csv("../../data/preprocessed/cleaned_text_airbnb_paris.csv", keep_default_na=False)
split = pd.read_csv(f"../../data/splits/split_airbnb_paris_seed{SEED}.csv")

df = base.merge(texts[["row_id"] + TEXT_COLS], on="row_id", how="inner")
tr = df["row_id"].isin(split.loc[split["split"] == "train", "row_id"]).values
assert list(df.columns) == list(base.columns) + TEXT_COLS
print("Zeilen:", len(df), "| Train:", int(tr.sum()))

## fastText auf Trainingstexten trainieren
- Korpus = alle Freitextzellen **der Trainingszeilen**

In [ ]:
corpus_path = f"/tmp/ft_corpus_airbnb_paris_seed{SEED}.txt"
corpus = pd.concat([df.loc[tr, c].astype(str) for c in TEXT_COLS]).str.replace(r"[\r\n]+", " ", regex=True)
with open(corpus_path, "w") as f:
    f.write("\n".join(corpus.tolist()))

ft = fasttext.train_unsupervised(corpus_path, model="skipgram", dim=100, thread=1)
name_emb = {c: ft.get_sentence_vector(c) for c in TEXT_COLS}  # cached, constant per column

## Alle Zeilen einbetten, Spaltennamen addieren, LayerNorm

In [ ]:
parts = []
for c in TEXT_COLS:
    cells = df[c].astype(str).str.replace(r"[\r\n]+", " ", regex=True).tolist()
    e = np.vstack([ft.get_sentence_vector(t) for t in cells]) + name_emb[c]
    e = (e - e.mean(axis=1, keepdims=True)) / (e.std(axis=1, keepdims=True) + 1e-6)
    parts.append(pd.DataFrame(e, columns=[f"{c}_emb_{i}" for i in range(e.shape[1])], index=df.index))

emb = pd.concat(parts, axis=1)
print("Embedding-Block:", emb.shape)

## PCA 30 & 100 – auf Train gefittet, speichern

In [ ]:
for n in (30, 100):
    pca = PCA(n_components=n, random_state=SEED).fit(emb.loc[tr])
    red = pca.transform(emb)
    print(f"explained variance ({n} comps):", round(pca.explained_variance_ratio_.sum(), 4))
    out = pd.concat([base.reset_index(drop=True),
                     pd.DataFrame(red, columns=[f"pca_{i}" for i in range(n)])], axis=1)
    out.to_csv(f"../../data/preprocessed/fast_text_pca{n}_airbnb_paris_seed{SEED}.csv", index=False)
    print("gespeichert:", out.shape)

## Verifikation

In [ ]:
assert out.isna().sum().sum() == 0
assert out["row_id"].is_unique